# 實踐項目 2：神經網絡數學基礎

## 🎯 項目目標

從零開始實現一個簡單的神經網絡，深入理解：
- 前向傳播的線性代數運算
- 激活函數的作用
- 損失函數的計算
- 反向傳播的微積分原理

## 📚 涵蓋知識點

- 矩陣乘法與向量運算
- 導數與鏈式法則
- 梯度下降優化
- 自動微分機制

## 🔧 環境準備

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# 設置隨機種子
torch.manual_seed(42)
np.random.seed(42)

print("✅ 環境準備完成！")

## 📊 任務 1：生成分類數據集

我們將生成一個簡單的二分類數據集（螺旋數據）。

In [ ]:
def generate_spiral_data(n_points=100, noise=0.2):
    """
    生成螺旋數據集
    
    參數:
        n_points: 每個類別的樣本數
        noise: 噪聲強度
    
    返回:
        X: 特徵 (n_samples, 2)
        y: 標籤 (n_samples,)
    """
    n = n_points
    X = np.zeros((n * 2, 2))
    y = np.zeros(n * 2, dtype=int)
    
    for class_id in range(2):
        ix = range(n * class_id, n * (class_id + 1))
        r = np.linspace(0.0, 1, n)  # 半徑
        t = np.linspace(class_id * 4, (class_id + 1) * 4, n) + np.random.randn(n) * noise  # 角度
        X[ix] = np.c_[r * np.sin(t * 2.5), r * np.cos(t * 2.5)]
        y[ix] = class_id
    
    return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long)

# 生成數據
X, y = generate_spiral_data(n_points=100)

# 可視化
plt.figure(figsize=(8, 8))
plt.scatter(X[y == 0, 0], X[y == 0, 1], c='skyblue', s=50, alpha=0.8, edgecolors='black', label='Class 0')
plt.scatter(X[y == 1, 0], X[y == 1, 1], c='salmon', s=50, alpha=0.8, edgecolors='black', label='Class 1')
plt.xlabel('Feature 1', fontsize=12)
plt.ylabel('Feature 2', fontsize=12)
plt.title('Spiral Dataset', fontsize=14, fontweight='bold')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.axis('equal')
plt.show()

print(f"數據集大小: {X.shape}")
print(f"類別分佈: Class 0: {(y == 0).sum()}, Class 1: {(y == 1).sum()}")

## 🧠 任務 2：手動實現前向傳播

### 2.1 理解神經網絡的數學結構

一個簡單的兩層神經網絡：

$$
\begin{align*}
z_1 &= W_1 x + b_1 \quad &\text{(線性變換)} \\
a_1 &= \sigma(z_1) \quad &\text{(激活函數)} \\
z_2 &= W_2 a_1 + b_2 \quad &\text{(線性變換)} \\
\hat{y} &= \text{softmax}(z_2) \quad &\text{(輸出層)}
\end{align*}
$$

其中 $\sigma$ 是激活函數（如 ReLU）。

In [ ]:
class SimpleNeuralNetwork:
    """
    手動實現的簡單神經網絡
    """
    
    def __init__(self, input_dim, hidden_dim, output_dim):
        """
        初始化網絡參數
        
        參數:
            input_dim: 輸入特徵維度
            hidden_dim: 隱藏層維度
            output_dim: 輸出維度（類別數）
        """
        # TODO: 初始化權重和偏置
        # 使用 He 初始化
        self.W1 = torch.randn(input_dim, hidden_dim) * np.sqrt(2.0 / input_dim)
        self.b1 = torch.zeros(hidden_dim)
        
        self.W2 = torch.randn(hidden_dim, output_dim) * np.sqrt(2.0 / hidden_dim)
        self.b2 = torch.zeros(output_dim)
        
        # 存儲中間值（用於反向傳播）
        self.cache = {}
    
    def relu(self, x):
        """ReLU 激活函數"""
        return torch.maximum(x, torch.zeros_like(x))
    
    def relu_derivative(self, x):
        """ReLU 導數"""
        return (x > 0).float()
    
    def softmax(self, x):
        """Softmax 函數（數值穩定版本）"""
        # TODO: 實現 softmax
        exp_x = torch.exp(x - x.max(dim=1, keepdim=True)[0])  # 數值穩定
        return exp_x / exp_x.sum(dim=1, keepdim=True)
    
    def forward(self, X):
        """
        前向傳播
        
        參數:
            X: 輸入數據 (batch_size, input_dim)
        
        返回:
            predictions: 預測概率 (batch_size, output_dim)
        """
        # TODO: 實現前向傳播
        # 第一層
        z1 = X @ self.W1 + self.b1  # 線性變換
        a1 = self.relu(z1)           # ReLU 激活
        
        # 第二層
        z2 = a1 @ self.W2 + self.b2  # 線性變換
        predictions = self.softmax(z2)  # Softmax 輸出
        
        # 保存中間值
        self.cache = {'X': X, 'z1': z1, 'a1': a1, 'z2': z2, 'predictions': predictions}
        
        return predictions
    
    def compute_loss(self, predictions, y):
        """
        計算交叉熵損失
        
        參數:
            predictions: 預測概率
            y: 真實標籤
        
        返回:
            loss: 標量損失值
        """
        # TODO: 實現交叉熵損失
        n = predictions.shape[0]
        # 選擇正確類別的概率
        correct_log_probs = -torch.log(predictions[range(n), y] + 1e-8)
        loss = correct_log_probs.mean()
        return loss

# 測試前向傳播
model = SimpleNeuralNetwork(input_dim=2, hidden_dim=10, output_dim=2)
predictions = model.forward(X)
loss = model.compute_loss(predictions, y)

print(f"前向傳播測試：")
print(f"預測形狀: {predictions.shape}")
print(f"初始損失: {loss.item():.4f}")
print(f"預測概率示例（前3個樣本）:\n{predictions[:3]}")

## 📐 任務 3：手動實現反向傳播

### 3.1 反向傳播的數學推導

使用鏈式法則計算梯度：

$$
\begin{align*}
\frac{\partial L}{\partial W_2} &= \frac{\partial L}{\partial z_2} \frac{\partial z_2}{\partial W_2} = a_1^T \delta_2 \\
\frac{\partial L}{\partial W_1} &= \frac{\partial L}{\partial z_1} \frac{\partial z_1}{\partial W_1} = x^T \delta_1
\end{align*}
$$

其中 $\delta$ 是誤差項。

In [ ]:
def backward(model, y):
    """
    反向傳播
    
    參數:
        model: SimpleNeuralNetwork 實例
        y: 真實標籤
    
    返回:
        gradients: 包含所有參數梯度的字典
    """
    # 獲取前向傳播的中間值
    X = model.cache['X']
    z1 = model.cache['z1']
    a1 = model.cache['a1']
    predictions = model.cache['predictions']
    
    n = X.shape[0]
    
    # TODO: 計算輸出層的梯度
    # dL/dz2（softmax + 交叉熵的梯度）
    delta2 = predictions.clone()
    delta2[range(n), y] -= 1  # softmax 梯度的簡化形式
    delta2 /= n
    
    # dL/dW2 和 dL/db2
    dW2 = a1.T @ delta2
    db2 = delta2.sum(dim=0)
    
    # TODO: 計算隱藏層的梯度（使用鏈式法則）
    # dL/da1
    delta1 = delta2 @ model.W2.T
    # dL/dz1 = dL/da1 * da1/dz1
    delta1 = delta1 * model.relu_derivative(z1)
    
    # dL/dW1 和 dL/db1
    dW1 = X.T @ delta1
    db1 = delta1.sum(dim=0)
    
    return {'dW1': dW1, 'db1': db1, 'dW2': dW2, 'db2': db2}

# 測試反向傳播
gradients = backward(model, y)

print("反向傳播測試：")
for name, grad in gradients.items():
    print(f"{name} 梯度形狀: {grad.shape}, 範數: {torch.norm(grad):.4f}")

### 🎯 練習 3.1：驗證梯度計算

使用數值梯度驗證我們的解析梯度是否正確。

數值梯度公式：
$$
\frac{\partial L}{\partial w} \approx \frac{L(w + \epsilon) - L(w - \epsilon)}{2\epsilon}
$$

In [ ]:
def numerical_gradient(model, X, y, param_name, epsilon=1e-5):
    """
    計算數值梯度
    
    參數:
        model: 模型
        X, y: 數據
        param_name: 參數名稱 ('W1', 'b1', 'W2', 'b2')
        epsilon: 微小擾動
    
    返回:
        數值梯度
    """
    param = getattr(model, param_name)
    grad = torch.zeros_like(param)
    
    # 對每個參數計算數值梯度（只計算一個元素作為示例）
    it = np.nditer(param.numpy(), flags=['multi_index'], op_flags=['readwrite'])
    
    while not it.finished:
        idx = it.multi_index
        old_value = param[idx].item()
        
        # f(x + epsilon)
        param[idx] = old_value + epsilon
        pred_plus = model.forward(X)
        loss_plus = model.compute_loss(pred_plus, y)
        
        # f(x - epsilon)
        param[idx] = old_value - epsilon
        pred_minus = model.forward(X)
        loss_minus = model.compute_loss(pred_minus, y)
        
        # 數值梯度
        grad[idx] = (loss_plus - loss_minus) / (2 * epsilon)
        
        # 恢復原值
        param[idx] = old_value
        it.iternext()
    
    return grad

# 驗證梯度（使用小批量數據）
X_small = X[:10]
y_small = y[:10]

model_test = SimpleNeuralNetwork(2, 5, 2)  # 使用更小的網絡以加快計算
_ = model_test.forward(X_small)
analytical_grads = backward(model_test, y_small)

print("梯度驗證（數值梯度 vs 解析梯度）：")
print("注意：由於計算量大，我們只驗證幾個元素")

# 驗證 W1 的幾個元素
numerical_grad_W1 = numerical_gradient(model_test, X_small, y_small, 'W1')
print(f"\nW1 梯度差異: {torch.norm(numerical_grad_W1 - analytical_grads['dW1']):.6f}")
print("（差異應該很小，< 1e-5）")

## 🎓 任務 4：實現梯度下降訓練

### 4.1 梯度下降更新規則

$$
W \leftarrow W - \eta \frac{\partial L}{\partial W}
$$

其中 $\eta$ 是學習率。

In [ ]:
def train(model, X, y, learning_rate=0.1, epochs=1000, verbose=True):
    """
    訓練神經網絡
    
    參數:
        model: SimpleNeuralNetwork
        X, y: 訓練數據
        learning_rate: 學習率
        epochs: 訓練輪數
        verbose: 是否打印訓練信息
    
    返回:
        loss_history: 損失歷史
        accuracy_history: 準確率歷史
    """
    loss_history = []
    accuracy_history = []
    
    for epoch in range(epochs):
        # 前向傳播
        predictions = model.forward(X)
        loss = model.compute_loss(predictions, y)
        
        # 反向傳播
        grads = backward(model, y)
        
        # TODO: 參數更新（梯度下降）
        model.W1 -= learning_rate * grads['dW1']
        model.b1 -= learning_rate * grads['db1']
        model.W2 -= learning_rate * grads['dW2']
        model.b2 -= learning_rate * grads['db2']
        
        # 計算準確率
        predicted_class = predictions.argmax(dim=1)
        accuracy = (predicted_class == y).float().mean()
        
        # 記錄
        loss_history.append(loss.item())
        accuracy_history.append(accuracy.item())
        
        # 打印進度
        if verbose and (epoch + 1) % 100 == 0:
            print(f"Epoch {epoch + 1}/{epochs}, Loss: {loss.item():.4f}, Accuracy: {accuracy.item():.4f}")
    
    return loss_history, accuracy_history

# 訓練模型
model = SimpleNeuralNetwork(input_dim=2, hidden_dim=20, output_dim=2)
loss_history, accuracy_history = train(model, X, y, learning_rate=0.5, epochs=2000)

print("\n✅ 訓練完成！")

### 可視化訓練過程

In [ ]:
# 繪製損失和準確率曲線
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 損失曲線
ax1.plot(loss_history, linewidth=2, color='royalblue')
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Training Loss', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)

# 準確率曲線
ax2.plot(accuracy_history, linewidth=2, color='green')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy', fontsize=12)
ax2.set_title('Training Accuracy', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.set_ylim([0, 1.05])

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"最終損失: {loss_history[-1]:.4f}")
print(f"最終準確率: {accuracy_history[-1]:.4f}")

### 可視化決策邊界

In [ ]:
def plot_decision_boundary(model, X, y):
    """
    繪製決策邊界
    """
    # 創建網格
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                        np.linspace(y_min, y_max, 200))
    
    # 預測網格點
    grid_points = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)
    with torch.no_grad():
        Z = model.forward(grid_points).argmax(dim=1).numpy()
    Z = Z.reshape(xx.shape)
    
    # 繪圖
    plt.figure(figsize=(10, 8))
    plt.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlBu', levels=1)
    plt.scatter(X[y == 0, 0], X[y == 0, 1], c='skyblue', s=60, 
               alpha=0.8, edgecolors='black', linewidth=1.5, label='Class 0')
    plt.scatter(X[y == 1, 0], X[y == 1, 1], c='salmon', s=60,
               alpha=0.8, edgecolors='black', linewidth=1.5, label='Class 1')
    
    plt.xlabel('Feature 1', fontsize=12)
    plt.ylabel('Feature 2', fontsize=12)
    plt.title('Decision Boundary', fontsize=14, fontweight='bold')
    plt.legend(fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.savefig('decision_boundary.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_decision_boundary(model, X, y)

## 🤖 任務 5：使用 PyTorch 自動微分對比

驗證我們手動實現的反向傳播是否正確。

In [ ]:
class PyTorchNN(nn.Module):
    """使用 PyTorch 自動微分的神經網絡"""
    
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# 訓練 PyTorch 版本
pytorch_model = PyTorchNN(2, 20, 2)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(pytorch_model.parameters(), lr=0.5)

pytorch_loss_history = []
pytorch_accuracy_history = []

for epoch in range(2000):
    # 前向傳播
    outputs = pytorch_model(X)
    loss = criterion(outputs, y)
    
    # 反向傳播
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    # 計算準確率
    with torch.no_grad():
        predicted = outputs.argmax(dim=1)
        accuracy = (predicted == y).float().mean()
    
    pytorch_loss_history.append(loss.item())
    pytorch_accuracy_history.append(accuracy.item())
    
    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch + 1}/2000, Loss: {loss.item():.4f}, Accuracy: {accuracy.item():.4f}")

print("\n✅ PyTorch 訓練完成！")

### 對比兩種實現

In [ ]:
# 對比訓練曲線
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 損失對比
ax1.plot(loss_history, linewidth=2, label='Manual Implementation', alpha=0.8)
ax1.plot(pytorch_loss_history, linewidth=2, label='PyTorch Autograd', alpha=0.8, linestyle='--')
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Loss Comparison', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 準確率對比
ax2.plot(accuracy_history, linewidth=2, label='Manual Implementation', alpha=0.8)
ax2.plot(pytorch_accuracy_history, linewidth=2, label='PyTorch Autograd', alpha=0.8, linestyle='--')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy', fontsize=12)
ax2.set_title('Accuracy Comparison', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_ylim([0, 1.05])

plt.tight_layout()
plt.savefig('comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n對比結果：")
print(f"手動實現 - 最終損失: {loss_history[-1]:.4f}, 準確率: {accuracy_history[-1]:.4f}")
print(f"PyTorch   - 最終損失: {pytorch_loss_history[-1]:.4f}, 準確率: {pytorch_accuracy_history[-1]:.4f}")
print("\n✅ 兩種實現的結果應該非常接近！")

## 📝 項目總結

### 你學到了什麼？

✅ 神經網絡的數學結構（矩陣乘法、激活函數）  
✅ 前向傳播的完整實現  
✅ 反向傳播和鏈式法則的應用  
✅ 梯度下降優化算法  
✅ 數值梯度驗證方法  
✅ PyTorch 自動微分機制  

### 關鍵數學概念

1. **矩陣乘法**: $y = Wx + b$
2. **鏈式法則**: $\frac{\partial L}{\partial W} = \frac{\partial L}{\partial y} \frac{\partial y}{\partial W}$
3. **梯度下降**: $W \leftarrow W - \eta \nabla_W L$
4. **Softmax**: $\sigma(z)_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$
5. **交叉熵**: $L = -\sum_i y_i \log(\hat{y}_i)$

### 🎯 進階挑戰

1. **動量優化器**: 實現帶動量的梯度下降
2. **批量訓練**: 實現 mini-batch 訓練
3. **正則化**: 添加 L2 正則化防止過擬合
4. **學習率調度**: 實現學習率衰減策略
5. **更深的網絡**: 擴展到 3 層或更多層

### 📚 推薦閱讀

- [CS231n: Backpropagation](http://cs231n.stanford.edu/slides/2022/lecture_4.pdf)
- [Deep Learning Book - Chapter 6](https://www.deeplearningbook.org/contents/mlp.html)
- [PyTorch Autograd Tutorial](https://pytorch.org/tutorials/beginner/blitz/autograd_tutorial.html)

---

**恭喜完成項目 2！🎉**

你已經深入理解了神經網絡的數學基礎！

繼續探索更多高級主題吧！